# PackingSolver exploration

the visualisation bit needs plotly and also nbformat>=4.2.0

In [1]:
import json
import sys
from importlib.metadata import version
from pathlib import Path

import packingsolver.rectangle as psr

print("Python:", sys.version)
print("PackingSolver:", version("packingsolver"))

Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
PackingSolver: 0.1.1340


Load one existing dataset

In [2]:
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

dataset_path = project_root / "data" / "synthetic" / "pilot_v1.json"

with dataset_path.open() as file:
    data = json.load(file)

datasets = data["datasets"]

In [3]:
dataset = next(
    dataset
    for dataset in datasets
    if dataset["dataset_id"]
    == "rect_v1_mixed_sheet2500x1250_n35_seed0"
)

parts = dataset["parts"]
sheet_w, sheet_h = dataset["sheet_size"]

print("Dataset:", dataset["dataset_id"])
print("Parts:", len(parts))
print("Sheet size:", (sheet_w, sheet_h))

Dataset: rect_v1_mixed_sheet2500x1250_n35_seed0
Parts: 35
Sheet size: (2500, 1250)


### Build the instance

- Using the plain bin-packing objective: place all requested parts while minimising the number of sheets. 
- PackingSolver distinguishes this from BinPackingWithLeftovers, which additionally considers the leftover on the last sheet. We should leave that extra objective out of the initial comparison.
- rotation not yet established

In [4]:
available_sheets = len(parts)

builder = psr.InstanceBuilder()
builder.set_objective(psr.Objective.BinPacking)

builder.add_bin_type(
    sheet_w,
    sheet_h,
    copies=available_sheets,
)

for part_id, width, height in parts:
    builder.add_item_type(
        width,
        height,
        copies=1,
    )

instance = builder.build()

### Run the solver

In [5]:
parameters = psr.OptimizeParameters()
parameters.time_limit = 5

output = psr.optimize(instance, parameters)

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  35
Number of items:       35
Number of bin types:   1
Number of bins:        35
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       12133094
Mean item width:       827.143
Mean item height:      420.943
Smallest item width:   139
Smallest item height:  139
Total item profit:     1.21331e+07
Largest item profit:   745240
Total item weight:     0
Largest item copies:   1
Total bin area:        109375000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
       0.160       5           22.35                  TS g 0 d X q 1

Final statistics
----------------
Time (s):              5.00589
Is proven infeasible:  0
Primal bound:          5
Dual bound:    

### Inspect and visualise the result

In [6]:
print(output)
print(output.solution)

Time (s):              5.00589
Is proven infeasible:  0
Primal bound:          5
Dual bound:            4
Gap:                   20%



In [7]:
psr.visualize(output.solution).show()